[//]: # (cr:doc name='chapter_c05_snapshot_and_dashboard' id=a5eb272a)
# Chapter c05: Snapshot + Dashboard (Causal Track)

Builds the per-scoring-run `eligibility_snapshot` table, publishes the six dashboard SQL views, and prints the four-way anchor tuple in force.

Reads `predictions` from `c04_batch_inference` (or the `s10_batch_inference` stage of the generated pipeline) — do **not** trigger scoring here. Reads the active rows from `archetype_catalog`, `eligibility_policy`, and `decision_policy`. Writes `eligibility_snapshot` via Delta MERGE on the natural `(scoring_run_id, account_id, playbook_id)` key — re-running with the same anchor tuple is a no-op.


In [ ]:
# @cr:code name='init_progress' id=43aad184
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("c05_snapshot_and_dashboard.ipynb")
# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


[//]: # (cr:doc name='c05_configuration' id=96b40ce2)
## Configuration

The cell below is the only place you should need to edit. Every value here is read by the snapshot writer and the dashboard publisher — nothing is hardcoded inside the algorithmic cells.

- **`SNAPSHOT_RISK_TIER_HIGH` / `SNAPSHOT_RISK_TIER_MEDIUM`** — risk-tier thresholds applied at snapshot time. Leave as `None` to fall back to the values stored on the active `decision_policy` row (the canonical source — set in `c01_publish_definitions`).
- **`SNAPSHOT_CAPACITY_PARTITION_COLUMN`** — optional partition column for capacity caps (e.g. `"csm_owner_id"`). Leave as `""` to apply caps globally per playbook.
- **`SHAP_PER_SLICE_K`** — for each `(playbook, archetype, risk_tier)` slice, compute per-row SHAP for the top-K accounts by expected loss and write them to `top_shap_drivers` so the dashboard's L4 panel can show "why surfaced" without a pandas_udf at view time. `0` disables the writer (the L3 view falls back gracefully but per-account SHAP cells stay empty for new model versions).
- **`SHAP_TOP_DRIVERS_PER_ROW`** — how many drivers to keep per row (default `5`).


In [ ]:
# @cr:config name='configuration' id=2586e099
SNAPSHOT_RISK_TIER_HIGH = None
SNAPSHOT_RISK_TIER_MEDIUM = None
SNAPSHOT_CAPACITY_PARTITION_COLUMN = ""
SHAP_PER_SLICE_K = 50
SHAP_TOP_DRIVERS_PER_ROW = 5

# === Optional engagement overrides — leave None for auto-detection ====
# RunNamespace.resolve() honours these first, then falls back to
# discovery tiers (CR_RUN_ID env, project pointer .cr_active_run.json,
# experiments_root/runs/.active_run_id sentinel, latest-by-mtime).
# Pin them when several runs share an experiments dir and auto-detection
# picked the wrong one.
ENGAGEMENT_RUN_ID = None
ENGAGEMENT_EXPERIMENTS_DIR = None
MODEL_URI_OVERRIDE = None  # if None, looked up from MLflow @production alias


[//]: # (cr:doc name='c05_snapshot_and_dashboard_setup' id=fcbd9d82)
## 5.0 Setup

Resolves catalog / schema / model identifiers from `ScoringConfig` (reads the persisted Databricks init JSON on Databricks, or the local pipeline's `best_model_meta.json` for local runs). The composite-name-qualified gold features table name is derived here so the algorithmic cells stay free of path-construction logic.


In [ ]:
# @cr:code name='setup_and_resolve_model' id=17068626
from customer_retention.core.compat.detection import get_spark_session, is_databricks
from customer_retention.core.config import get_playbooks_dir
from customer_retention.stages.scoring import resolve_scoring_context

spark = get_spark_session()
PLAYBOOKS_DIR = get_playbooks_dir()

# Auto-detect the active run / model via RunNamespace's file-tracked
# discovery tiers (CR_RUN_ID env, project pointer .cr_active_run.json,
# experiments_root/runs/.active_run_id sentinel, latest-by-mtime). On
# Databricks, the model URI is resolved from MLflow's @production alias
# for the registered model named in training_metadata.json. Operator
# overrides from the configuration cell above (ENGAGEMENT_RUN_ID /
# ENGAGEMENT_EXPERIMENTS_DIR / MODEL_URI_OVERRIDE) short-circuit each
# resolution tier when set; default None → auto-detect.
_resolved = resolve_scoring_context(
    run_id=globals().get("ENGAGEMENT_RUN_ID"),
    experiments_dir=globals().get("ENGAGEMENT_EXPERIMENTS_DIR"),
    model_uri=globals().get("MODEL_URI_OVERRIDE"),
)
scoring_config = _resolved.scoring_config
_namespace = _resolved.namespace
_ns_source = _resolved.source
CATALOG = scoring_config.catalog if is_databricks() else "local"
SCHEMA = scoring_config.schema if is_databricks() else "local"
MODEL_NAME = _resolved.model_name
MODEL_VERSION = _resolved.model_version
MODEL_URI = _resolved.model_uri

COMPOSITE_NAME = scoring_config.composite_name
GOLD_FEATURES_FQN = (
    f"{CATALOG}.{SCHEMA}.gold_features_{COMPOSITE_NAME}"
    if COMPOSITE_NAME
    else f"{CATALOG}.{SCHEMA}.gold_features"
)

ARCHETYPE_CATALOG_FQN = f"{CATALOG}.{SCHEMA}.archetype_catalog"
ELIGIBILITY_POLICY_FQN = f"{CATALOG}.{SCHEMA}.eligibility_policy"
DECISION_POLICY_FQN = f"{CATALOG}.{SCHEMA}.decision_policy"
ELIGIBILITY_SNAPSHOT_FQN = f"{CATALOG}.{SCHEMA}.eligibility_snapshot"
PREDICTIONS_FQN = f"{CATALOG}.{SCHEMA}.predictions"
TOP_SHAP_DRIVERS_FQN = f"{CATALOG}.{SCHEMA}.top_shap_drivers"

print(f"Resolved playbooks_dir: {PLAYBOOKS_DIR}")
print(f"Active run namespace:   {_namespace.run_id if _namespace else '(none)'}")
print(f"Run source:             {_ns_source}")
print(f"Catalog/schema:         {CATALOG}.{SCHEMA}")
print(f"Composite name:         {COMPOSITE_NAME or '(unset)'}")
print(f"Gold features table:    {GOLD_FEATURES_FQN}")
print(f"Model URI:              {MODEL_URI or '(local)'}")
print(f"Model version:          {MODEL_VERSION}")


[//]: # (cr:doc name='c05_build_snapshot_section' id=557c5c9f)
## 5.1 Build Eligibility Snapshot


In [ ]:
# @cr:code name='build_eligibility_snapshot' id=345cddf0
from customer_retention.stages.causal import SnapshotConfig, build_eligibility_snapshot

snapshot_result = None
if spark is None:
    print("SKIPPED: no Spark session (Databricks-only cell)")
elif not spark.catalog.tableExists(ARCHETYPE_CATALOG_FQN):
    print(f"SKIPPED: {ARCHETYPE_CATALOG_FQN} does not exist (run c01..c03 first)")
elif not spark.catalog.tableExists(PREDICTIONS_FQN):
    print(f"SKIPPED: {PREDICTIONS_FQN} not populated (run c04_batch_inference first)")
else:
    snapshot_cfg = SnapshotConfig(
        spark=spark,
        predictions_fqn=PREDICTIONS_FQN,
        archetype_catalog_fqn=ARCHETYPE_CATALOG_FQN,
        eligibility_policy_fqn=ELIGIBILITY_POLICY_FQN,
        decision_policy_fqn=DECISION_POLICY_FQN,
        snapshot_table_fqn=ELIGIBILITY_SNAPSHOT_FQN,
        model_name=MODEL_NAME,
        model_version=MODEL_VERSION,
        # gold_features_fqn is the raw-feature source for predicate evaluation
        # (eligibility rules like `active_span_days >= 42` reference raw
        # feature columns that are NOT carried on the predictions table).
        gold_features_fqn=GOLD_FEATURES_FQN,
        # entity_id_column names the scoring-subject key on predictions, gold,
        # and the snapshot output. Default is "entity_id" — override only if
        # your predictions table keys on a different column (e.g. account_id).
        entity_id_column="entity_id",
        risk_tier_high=SNAPSHOT_RISK_TIER_HIGH,
        risk_tier_medium=SNAPSHOT_RISK_TIER_MEDIUM,
        capacity_partition_column=SNAPSHOT_CAPACITY_PARTITION_COLUMN or None,
        top_shap_drivers_fqn=TOP_SHAP_DRIVERS_FQN or None,
    )
    snapshot_result = build_eligibility_snapshot(snapshot_cfg)
    print(snapshot_result.summary())


[//]: # (cr:doc name='c05_compute_top_shap_section' id=e7d89d6f)
## 5.2 Compute Per-slice Top SHAP Drivers


In [ ]:
# @cr:code name='compute_top_shap_drivers' id=0fd15e69
from customer_retention.stages.causal import (
    TopDriversConfig,
    compute_and_write_top_shap_drivers,
)

# Per-slice SHAP cache for the L4 "why surfaced" panel. Keyed on
# (model_name, model_version, entity_id) so v_eligible_all_playbooks can
# fall back to it via COALESCE when the snapshot row's top_shap_features
# column is NULL. Without this writer, every new model_version produces
# 0 rows here and the dashboard's L3 cohort list comes back empty (the
# view filters WHERE top_shap_features IS NOT NULL after the COALESCE).
top_drivers_result = None
if spark is None:
    print("SKIPPED: no Spark session (Databricks-only cell)")
elif snapshot_result is None:
    print("SKIPPED: snapshot_result is None — 5.1 did not produce a run")
elif int(SHAP_PER_SLICE_K) <= 0:
    print("SKIPPED: SHAP_PER_SLICE_K == 0 (per-slice SHAP enrichment disabled)")
elif MODEL_URI is None:
    print("SKIPPED: MODEL_URI is None — local runs without an MLflow attribution artifact cannot replay SHAP")
elif not spark.catalog.tableExists(GOLD_FEATURES_FQN):
    print(f"SKIPPED: {GOLD_FEATURES_FQN} does not exist — gold features required for SHAP replay")
else:
    top_drivers_cfg = TopDriversConfig(
        spark=spark,
        snapshot_table_fqn=ELIGIBILITY_SNAPSHOT_FQN,
        gold_features_fqn=GOLD_FEATURES_FQN,
        top_shap_drivers_fqn=TOP_SHAP_DRIVERS_FQN,
        model_name=MODEL_NAME,
        model_version=str(MODEL_VERSION),
        scoring_run_id=snapshot_result.scoring_run_id,
        as_of_date=snapshot_result.as_of_date,
        model_uri=MODEL_URI,
        per_slice_k=int(SHAP_PER_SLICE_K),
        top_drivers_per_row=int(SHAP_TOP_DRIVERS_PER_ROW),
    )
    top_drivers_result = compute_and_write_top_shap_drivers(top_drivers_cfg)
    print(top_drivers_result.summary())


[//]: # (cr:doc name='c05_write_run_context_section' id=9c2e5465)
## 5.3 Write Run Context (app masthead projection)


In [ ]:
# @cr:code name='write_run_context' id=9a423d80
from customer_retention.stages.causal import (
    from_project_context,
    write_run_context,
)

RUN_CONTEXT_FQN = f"{CATALOG}.{SCHEMA}.run_context"


def _load_project_context():
    """Best-effort ProjectContext load. Returns None when YAML is unreachable so
    the writer still emits a row with model metadata only."""
    try:
        from customer_retention.analysis.auto_explorer.project_context import ProjectContext
        from customer_retention.analysis.auto_explorer.run_namespace import RunNamespace
    except ImportError:
        return None
    try:
        _ns = RunNamespace.from_env_or_latest()
    except Exception:
        _ns = None
    if _ns is None:
        return None
    _path = _ns.project_context_path
    if not _path.exists():
        return None
    try:
        return ProjectContext.load(_path)
    except Exception:
        return None


def _resolve_model_type():
    """Try to pull the MLflow flavor off the registered model (e.g. "xgboost")."""
    try:
        import mlflow  # noqa: F401
        if MODEL_URI is None:
            return None
        from mlflow.models import Model as _MlflowModel
        _info = _MlflowModel.load(MODEL_URI)
        _flavors = list((_info.flavors or {}).keys())
        for preferred in ("xgboost", "lightgbm", "catboost", "pytorch", "tensorflow", "sklearn"):
            if preferred in _flavors:
                return preferred
        _flavors = [f for f in _flavors if f != "python_function"]
        return _flavors[0] if _flavors else None
    except Exception:
        return None


if spark is None:
    print("SKIPPED: no Spark session (Databricks-only cell)")
elif snapshot_result is None:
    print("SKIPPED: snapshot_result is None — 5.1 did not produce a run")
else:
    _ctx = _load_project_context()
    _cfg = from_project_context(
        project_context=_ctx,
        spark=spark,
        table_fqn=RUN_CONTEXT_FQN,
        scoring_run_id=snapshot_result.scoring_run_id,
        as_of_date=snapshot_result.as_of_date,
        model_name=MODEL_NAME,
        model_version=MODEL_VERSION,
        model_type=_resolve_model_type(),
    )
    write_run_context(_cfg)
    print(f"Wrote run_context row for scoring_run_id={snapshot_result.scoring_run_id}")
    if _ctx is None:
        print("  (project_context.yaml not reachable — context fields are NULL)")
    else:
        print(f"  horizon_days:       {_cfg.horizon_days}")
        print(f"  primary_objective:  {_cfg.primary_objective}")
        print(f"  temporal_posture:   {_cfg.temporal_posture}")
        print(f"  model_type:         {_cfg.model_type or '(unresolved)'}")


[//]: # (cr:doc name='c05_publish_views_section' id=a6cbe88f)
## 5.4 Publish Dashboard SQL Views


In [ ]:
# @cr:code name='publish_dashboard_views' id=00cf348e
from customer_retention.stages.causal.dashboard_views import (
    DASHBOARD_VIEW_NAMES,
    publish_dashboard_views,
)

if spark is None:
    print("SKIPPED: no Spark session (Databricks-only cell)")
elif not spark.catalog.tableExists(ELIGIBILITY_SNAPSHOT_FQN):
    print(f"SKIPPED: {ELIGIBILITY_SNAPSHOT_FQN} not populated yet")
elif not COMPOSITE_NAME:
    print("SKIPPED: COMPOSITE_NAME is unset — deviation views require it. Re-run cell 5 (setup_and_resolve_model) so scoring_config resolves COMPOSITE_NAME, or set it manually in the configuration cell above.")
else:
    statements = publish_dashboard_views(
        spark,
        CATALOG,
        SCHEMA,
        composite_name=COMPOSITE_NAME,
    )
    print(f"Published {len(statements)} dashboard views (composite_name={COMPOSITE_NAME!r}):")
    for view_name in DASHBOARD_VIEW_NAMES:
        print(f"  - {CATALOG}.{SCHEMA}.{view_name}")


[//]: # (cr:doc name='c05_summary_section' id=fd17e5f0)
## 5.5 Print Run Summary


In [ ]:
# @cr:code name='print_run_summary' id=b2232bcf
if spark is None or not spark.catalog.tableExists(ARCHETYPE_CATALOG_FQN):
    print("(no archetype_catalog yet — run c01..c03 first)")
else:
    counts = spark.sql(
        f"SELECT status, COUNT(*) AS n FROM {ARCHETYPE_CATALOG_FQN} GROUP BY status"
    ).collect()
    print("archetype_catalog row counts:")
    for row in counts:
        print(f"  {row['status']}: {row['n']}")
    print(f"Model: {MODEL_NAME} v{MODEL_VERSION}")
    if snapshot_result is not None:
        print(snapshot_result.summary())


In [ ]:
# @cr:code name='release_stage_memory' id=c9bb28b6
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
